# Drone RL — results analysis

Loads the artefacts produced by `scripts/train.py` and `scripts/evaluate.py` and
reproduces the headline numbers and figures.

Run the pipeline first:

```bash
python scripts/train.py    --config configs/dqn.yaml
python scripts/evaluate.py --checkpoint outputs/checkpoints/dqn_best.pt --episodes 100 --with-random
```

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

OUTPUTS = PROJECT_ROOT / "outputs"
pd.set_option("display.width", 140)
print("project root:", PROJECT_ROOT)

## 1. Training curves

In [ ]:
runs = sorted((OUTPUTS / "logs").glob("*/metrics.csv"))
print("runs found:", [p.parent.name for p in runs])

metrics = pd.read_csv(runs[-1])
print(f"{len(metrics)} episodes from {runs[-1].parent.name}")
metrics.tail()

In [ ]:
# Progress in blocks of 200 episodes: the clearest view of what was learned.
block = 200
blocks = metrics.assign(block=(metrics.episode - 1) // block * block + 1).groupby("block").agg(
    success=("success", "mean"),
    collision=("collision", "mean"),
    timeout=("timeout", "mean"),
    out_of_bounds=("out_of_bounds", "mean"),
    reward=("reward", "mean"),
    steps=("steps", "mean"),
    energy=("energy", "mean"),
)
for column in ["success", "collision", "timeout", "out_of_bounds"]:
    blocks[column] = (100 * blocks[column]).round(1)
blocks.round(2)

In [ ]:
window = 100
fig, axes = plt.subplots(2, 2, figsize=(13, 7))

specs = [
    ("reward", "Episode reward", "#2A6FDB", 1.0),
    ("success", "Success rate (%)", "#2BA84A", 100.0),
    ("collision", "Collision rate (%)", "#D64550", 100.0),
    ("steps", "Episode length (steps)", "#E8833A", 1.0),
]
for ax, (column, title, colour, scale) in zip(axes.ravel(), specs):
    series = pd.to_numeric(metrics[column], errors="coerce") * scale
    ax.plot(metrics.episode, series, color="#AAB2BD", alpha=0.3, lw=0.7)
    ax.plot(metrics.episode, series.rolling(window, min_periods=1).mean(), color=colour, lw=2)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("episode")
    ax.grid(alpha=0.25, ls="--")
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()

## 2. Validation history

Sweeps run during training on the **validation** seed pool (cities never trained on).
`dqn_best.pt` is the checkpoint with the highest validation success rate.

In [ ]:
evaluations_path = runs[-1].parent / "evaluations.json"
if evaluations_path.exists():
    history = pd.DataFrame(json.loads(evaluations_path.read_text()))
    columns = ["episode", "success_rate", "collision_rate", "timeout_rate", "mean_reward"]
    display(history[columns].round(3))

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(history.episode, 100 * history.success_rate, "o-", color="#2BA84A", label="success")
    ax.plot(history.episode, 100 * history.collision_rate, "o-", color="#D64550", label="collision")
    ax.set_xlabel("training episode")
    ax.set_ylabel("validation rate (%)")
    ax.legend(frameon=False)
    ax.grid(alpha=0.25, ls="--")
    ax.spines[["top", "right"]].set_visible(False)
else:
    print("No evaluations.json — run training first.")

## 3. Generalisation: training cities vs unseen cities

The research-relevant number. Seed pools are disjoint, so `test` layouts were
provably never trained on.

In [ ]:
rows = []
for path in sorted((OUTPUTS / "evaluation").glob("*_summary.json")):
    payload = json.loads(path.read_text())
    for policy, per_pool in payload.get("summaries", {}).items():
        for pool, stats in per_pool.items():
            rows.append({"policy": policy, "pool": pool, **stats})

if rows:
    summary = pd.DataFrame(rows)
    view = summary[[
        "policy", "pool", "success_rate", "collision_rate", "timeout_rate",
        "mean_reward", "mean_steps", "mean_energy", "mean_path_length",
    ]].copy()
    for column in ["success_rate", "collision_rate", "timeout_rate"]:
        view[column] = (100 * view[column]).round(1)
    display(view.round(2).sort_values(["policy", "pool"]))

    gap = summary.pivot_table(index="policy", columns="pool", values="success_rate")
    if {"train", "test"} <= set(gap.columns):
        gap["gap_pp"] = (100 * (gap["test"] - gap["train"])).round(1)
        display(gap.round(3))
else:
    print("No evaluation summaries — run scripts/evaluate.py first.")

## 4. Where do failures happen?

In [ ]:
episode_files = sorted((OUTPUTS / "evaluation").glob("*_episodes.csv"))
if episode_files:
    episodes = pd.concat([pd.read_csv(p) for p in episode_files], ignore_index=True)
    print(episodes.groupby(["policy", "pool"]).status.value_counts().unstack(fill_value=0))

    # Does the straight-line distance predict failure?
    fig, ax = plt.subplots(figsize=(8, 4))
    for status, group in episodes.groupby("status"):
        ax.scatter(group.straight_line_distance, group.reward, s=14, alpha=0.6, label=status)
    ax.set_xlabel("start-to-target distance (m)")
    ax.set_ylabel("episode reward")
    ax.legend(frameon=False, fontsize=8)
    ax.grid(alpha=0.25, ls="--")
    ax.spines[["top", "right"]].set_visible(False)
else:
    print("No per-episode CSVs — run scripts/evaluate.py first.")

## 5. Reward decomposition for a single episode

`info` exposes every reward component, which is how you diagnose *why* the
agent behaves as it does rather than guessing.

In [ ]:
import torch

from drone_rl.agents.registry import agent_from_checkpoint
from drone_rl.environment.drone_env import DroneCityEnv
from drone_rl.utils.config import load_config

checkpoint_path = OUTPUTS / "checkpoints" / "dqn_best.pt"
config = load_config(PROJECT_ROOT / "configs" / "environment.yaml")
env = DroneCityEnv(config=config, seed_pool="test")

if checkpoint_path.exists():
    payload = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    agent = agent_from_checkpoint(payload, device="cpu")
    agent.eval_mode()

    observation, _ = env.reset(seed=4242)
    trace, done = [], False
    while not done:
        observation, reward, terminated, truncated, info = env.step(
            agent.act(observation, evaluation=True)
        )
        trace.append({"step": info["step"], "distance": info["distance_to_target"],
                      "battery": info["battery"], **info["reward_components"]})
        done = terminated or truncated

    trace = pd.DataFrame(trace)
    print("outcome:", env.status, "| total reward:", round(env.episode_reward, 2))

    fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    for column in ["progress_reward", "proximity_penalty", "step_penalty", "energy_cost"]:
        axes[0].plot(trace.step, trace[column].cumsum(), label=column)
    axes[0].set_ylabel("cumulative reward")
    axes[0].legend(frameon=False, fontsize=8, ncol=2)
    axes[1].plot(trace.step, trace.distance, color="#2A6FDB")
    axes[1].axhline(env.env_config.target_radius, ls="--", color="#2BA84A", label="target radius")
    axes[1].set_ylabel("distance to target (m)")
    axes[1].set_xlabel("step")
    axes[1].legend(frameon=False, fontsize=8)
    for ax in axes:
        ax.grid(alpha=0.25, ls="--")
        ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
else:
    print("No checkpoint yet — run scripts/train.py first.")
env.close()